# 學生講義：遊戲選角（平方和取第二大）
Status: in_use  
Prereq: 對應主題的基礎語法（搜尋/遞迴/樹/圖等）  
Can-do checklist:  
- [ ] 能說出本主題的核心概念與用途
- [ ] 能完成至少 2 個小練習或 TODO
- [ ] 能指出 1 個常見錯誤並避免它


> 目標：學會將角色 (攻擊, 防禦) 轉成綜合能力 `power = atk^2 + def^2`，並用兩種演算法找出第二高的角色，同時維持輸出索引 (攻、防)。

## 本講義怎麼用？
1. 先閱讀題目摘要與暖身問題，確認理解輸入/輸出。
2. 跟著 Step 1~3 完成 TODO 程式碼，並用提供的測資把輸出與參考答案逐一對照。
3. 於每個 **思考記錄** 區塊寫下你的觀察或遇到的 bug。
4. 完成最後的延伸練習（top-3 或同分處理）並在 Notebook 內標記完成。


## 題目摘要
- 輸入：`n` 名角色 (3 ≤ n ≤ 20)，接著 n 行 `atk_i def_i`。
- 綜合能力：`power = atk_i^2 + def_i^2`，題目保證沒有同分。
- 輸出：綜合能力第二高角色的攻擊與防禦值。

### 輸入 / 輸出格式
- **輸入**：
  - 第一行 `n`，代表角色數量 (3 ≤ n ≤ 20)。
  - 接下來 n 行，每行兩個整數 `atk`、`def`，範圍 1~100。
- **輸出**：
  - 一行兩個整數，為綜合能力第二高角色的攻擊與防禦，中間以一個空白隔開。

範例：
```
輸入:
3
3 2
5 2
1 4

輸出:
1 4
```



### Step 0：讀取輸入並建立 `players` 清單
- 常見寫法：`n = int(input().strip())`、`atk, df = map(int, input().split())`。
- 由於 APCS 題目使用標準輸入，建議把讀取流程包成函式，方便日後測試。


In [74]:

# 範例：將文字輸入轉成 players list
sample_input = """
3
3 2
5 2
1 4
""".strip().splitlines()

n = int(sample_input[0])
players = []
for line in sample_input[1:1+n]:
    atk, df = map(int, line.split())
    players.append((atk, df))

print('n =', n)
print('players =', players)



n = 3
players = [(3, 2), (5, 2), (1, 4)]


In [75]:

# TODO：撰寫 read_players(lines) -> List[Tuple[int, int]]
# 1. lines 參數是一個字串列表
# 2. 第一行是 n，其餘是 atk def
# 3. 回傳 players list

def read_players(lines):
    pass

print('測試 read_players：', read_players(["3", "3 2", "5 2", "1 4"]))



測試 read_players： None


## 暖身問題（可先口頭回答）
1. 若輸入 `(3, 2), (5, 2), (1, 4)`，哪一位的 power 最大？第二大？
2. 若把資料照 power 由大到小排序，要取哪個索引才是第二大？
3. 若不排序，如何在單趟迴圈中同步記住第一名與第二名？需要哪些變數？

## 語法 & 工具複習
- `lambda` 與 `sorted(..., key=..., reverse=True)`：自訂排序鍵。
- `math.inf` / `-math.inf`：初始化最大/最小值。
- tuple 拆封：`for atk, df in players:`。
- 輸出比對：撰寫小測試列印「實際值 vs 預期值」。



### Lambda 語法快速複習
- `lambda 參數: 表達式` 會建立一個匿名函式，可立即拿來當排序鍵或 map/filter 函式。
- 例如 `lambda pair: pair[0]**2 + pair[1]**2` 代表：接受 `(atk, def)`，回傳 `atk^2 + def^2`。
- 可先把 lambda 指派給變數（`score = lambda pair: ...`），或直接放在呼叫處。


In [76]:

# 範例：lambda vs 一般函式

def square_sum(pair):
    atk, df = pair
    return atk * atk + df * df

lambda_square_sum = lambda pair: pair[0] ** 2 + pair[1] ** 2

sample_pair = (3, 2)
print('square_sum:', square_sum(sample_pair))
print('lambda_square_sum:', lambda_square_sum(sample_pair))



square_sum: 13
lambda_square_sum: 13


### 語法鋪墊 1：`sorted` + `lambda`
- `sorted(data, key=..., reverse=True)` 會回傳新的 list，不會修改原資料。
- `lambda pair: pair[0]**2 + pair[1]**2` 可以當作排序鍵。

In [77]:
# 範例：使用 sorted + lambda 依照 power 排序
players_demo = [(3, 2), (5, 2), (1, 4)]
def power(pair):
    atk, df = pair
    return atk * atk + df * df
ranked_demo = sorted(players_demo, key=power, reverse=True)
print('依 power 由大到小排序：')
for atk, df in ranked_demo:
    print(f'atk={atk}, def={df}, power={power((atk, df))}')


依 power 由大到小排序：
atk=5, def=2, power=29
atk=1, def=4, power=17
atk=3, def=2, power=13


In [78]:
# 練習：完成 sort_by_power(data) 並印出結果
players_demo = [(6, 1), (2, 5), (4, 3)]

def sort_by_power(data):
    """回傳依 power 由大到小排序的新串列"""
    # TODO：呼叫 sorted 並回傳結果
    pass

ranked = sort_by_power(players_demo)
print('練習輸出：', ranked)


練習輸出： None


### 語法鋪墊 2：tuple 拆封 + dict 建構
- 透過 `for atk, df in players_demo:` 同步取得兩個值。
- 可將結果組成 dict，方便加入衍生欄位。

In [79]:
# 範例：tuple 拆封並建立帶有 power 的 dict
players_demo = [(3, 2), (5, 2)]
annotated = []
for atk, df in players_demo:
    annotated.append({'atk': atk, 'def': df, 'power': atk * atk + df * df})
print(annotated)


[{'atk': 3, 'def': 2, 'power': 13}, {'atk': 5, 'def': 2, 'power': 29}]


In [80]:
# 練習：把新的 players 轉換為帶 power 的 dict list
players_demo = [(2, 7), (4, 1), (5, 5)]
annotated_demo = []
for _pair in players_demo:
    # TODO：拆封 atk/def、計算 power，append 到 annotated_demo
    pass
print('練習輸出：', annotated_demo)


練習輸出： []


### 語法鋪墊 3：`math.inf` / `-math.inf`
- 初始化最大值時使用 `-math.inf`，確保任何正數都比它大。
- 練習撰寫簡單的最大值追蹤程式。

In [81]:
# 範例：使用 -inf 追蹤最大值
from math import inf
numbers = [5, 3, 9, 1]
max_value = -inf
for value in numbers:
    if value > max_value:
        max_value = value
print('最大值 =', max_value)


最大值 = 9


In [82]:
# 練習：把 numbers2 的最大值找出來
from math import inf
numbers2 = [12, 7, 20, 4]
max_value = -inf
# TODO：寫出更新邏輯，最後印出 max_value




### Power 函式抽象化
- 將 `atk^2 + def^2` 寫成函式 `calc_power(atk, df)`，可避免重複程式碼。
- 後續排序與線性掃描都能共用這個函式。


In [83]:

# 範例：calc_power 函式

def calc_power(atk, df):
    return atk * atk + df * df

for pair in players_sample:
    print(pair, '=>', calc_power(*pair))



(3, 2) => 13
(5, 2) => 29
(1, 4) => 17
(4, 5) => 41


In [84]:

# TODO：呼叫 calc_power 完成列表推導，產生 [(atk, def, power)]
annotated_via_func = []  # 請改成實際的列表推導或迴圈
print('使用 calc_power 的結果：', annotated_via_func)



使用 calc_power 的結果： []


## Step 1：先計算 `power`（衍生欄位）
把每個 `(atk, def)` 轉成 `{'atk': ..., 'def': ..., 'power': ...}`，之後比較或排序就不用重複算平方。

In [85]:
# 範例資料，可自行修改測試
players_sample = [(3, 2), (5, 2), (1, 4), (4, 5)]
players_sample


[(3, 2), (5, 2), (1, 4), (4, 5)]

In [86]:
# TODO：完成 compute_power_list(players)
# 1. 回傳 list，每個元素是 dict，包含 atk、def、power
# 2. power = atk*atk + def*def
# 3. 先用 players_sample 測試並觀察輸出格式

from typing import List, Dict, Tuple

def compute_power_list(players: List[Tuple[int, int]]):
    # TODO：建立 result list 並加入每一筆資料
    pass

compute_power_list(players_sample)


> **思考記錄**：描述你如何確認計算正確？有沒有把 `power` 寫成函式以利重用？

## Step 2：方法一 — 排序法（O(n log n)）
1. 先計算 `power`。
2. 使用 `sorted` 依 power 由大到小排序。
3. 回傳排序後索引 1 的角色 (第二名)。

> 小提醒：別忘了保留 `(atk, def)`，而不是只回傳 power。

In [87]:
# 範例資料，可自行修改測試
players_sample = [(3, 2), (5, 2), (1, 4), (4, 5)]
print('參考 power 結果：', [atk*atk + df*df for atk, df in players_sample])
players_sample


參考 power 結果： [13, 29, 17, 41]


[(3, 2), (5, 2), (1, 4), (4, 5)]

> **思考記錄**：如果改為 `players.sort(...)` 會改變原本輸入嗎？在什麼情況要改用 `sorted`？

## Step 3：方法二 — 線性掃描（O(n)）
維護兩組變數：
- `first_power` / `first_role`：目前最大值及對應角色。
- `second_power` / `second_role`：目前第二大。

流程：
1. 逐一計算 `power`。
2. 若 `power > first_power`，先把第一名丟到第二名，再更新第一名。
3. 否則若 `power > second_power`，更新第二名。


### 線性掃描更新邏輯詳解
| 角色 | power | first_power/second_power 更新 | 備註 |
| --- | --- | --- | --- |
| `(3, 2)` | 13 | first=13, second=-inf | 第一筆自動成為最大值 |
| `(5, 2)` | 29 | first=29 (舊 first 變 second=13) | 新值大於 first，先下移後更新 |
| `(1, 4)` | 17 | second=17 | 介於 first、second 之間 |
| `(4, 5)` | 41 | first=41, second=29 | 再次更新 top-2 |
- 每次比較順序：先判斷是否超越 first，再判斷是否介於 first 與 second 之間。
- 記得同步更新 `first_role`、`second_role`，以保留 `(atk, def)`。


In [88]:

# 線性掃描步驟示範
sample_trace = [(3, 2), (5, 2), (1, 4), (4, 5)]
from math import inf

first_power = -inf
second_power = -inf
first_role = None
second_role = None

for atk, df in sample_trace:
    power = atk * atk + df * df
    print(f'目前角色 {atk, df} -> power={power}')
    if power > first_power:
        second_power, second_role = first_power, first_role
        first_power, first_role = power, (atk, df)
        print('  更新 first ->', first_role, '| second ->', second_role)
    elif power > second_power:
        second_power, second_role = power, (atk, df)
        print('  更新 second ->', second_role)
    else:
        print('  不更新，仍維持 top-2')

print('最終第一名、第二名：', first_role, second_role)



目前角色 (3, 2) -> power=13
  更新 first -> (3, 2) | second -> None
目前角色 (5, 2) -> power=29
  更新 first -> (5, 2) | second -> (3, 2)
目前角色 (1, 4) -> power=17
  更新 second -> (1, 4)
目前角色 (4, 5) -> power=41
  更新 first -> (4, 5) | second -> (5, 2)
最終第一名、第二名： (4, 5) (5, 2)


In [89]:
# TODO：完成 second_player_scan(players)
# - 使用單趟迴圈維護第一名/第二名
# - 回傳 (atk, def)

from math import inf

def second_player_scan(players):
    # TODO：宣告 first_power/second_power 等變數
    pass

print('線性掃描結果：', second_player_scan(sample))
# 參考答案：sample 應輸出 (4, 5)


NameError: name 'sample' is not defined

### 驗證兩種方法輸出一致

In [ ]:
# TODO：將 sample 改為多組測試，確保兩法相同
cases = [
    [(3, 2), (5, 2), (1, 4), (4, 5)],
    [(1, 1), (2, 3), (4, 2), (0, 5)],
]
for players in cases:
    sort_result = second_player_sort(players)
    scan_result = second_player_scan(players)
    print('案例:', players)
    print('  排序法=', sort_result, '| 掃描法=', scan_result)
    if sort_result != scan_result:
        print('  ⚠️ 結果不一致，請檢查邏輯！')



## 延伸練習（TODO）
1. **Top-3**：把線性掃描改寫成維護前三名，輸出前三名的 `(atk, def)`。
2. **同分處理**：假設題目不保證互異，請設計 tie-break 規則（例如同分時輸出攻擊較高者）。
3. **即時資料流**：寫一個函式 `update_top2(top2, new_player)`，每讀入一個新角色就更新 top-2。

## 自我檢核
- [ ] 能口頭說出排序法與線性掃描法的時間複雜度。
- [ ] 能畫出線性掃描更新順序。
- [ ] 能把兩種方法的輸出列印並確認一致。
- [ ] 能描述若允許同分，需要額外考慮哪些欄位。

> 完成後，把此 Notebook 與心得上傳至練習紀錄，寫下哪一種方法讓你更有信心，以及下一步要挑戰的延伸題。
